<a href="https://colab.research.google.com/github/Jamiul-kawsar/covid_chest_x-ray_curated_dataset/blob/main/notebooks/COVID_Chest_X_Ray_Research_Project_Curated_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import Path
from pathlib import Path

# Dataset path
DATA = Path(
    "/content/drive/MyDrive/Colab Notebooks/Curated COVID-19 Chest X-Ray Dataset"
)

# Check dataset
print("Dataset loaded successfully!")
print("Dataset path:", DATA)
print("Exists:", DATA.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully!
Dataset path: /content/drive/MyDrive/Colab Notebooks/Curated COVID-19 Chest X-Ray Dataset
Exists: True


**Dataset Audit**

In [5]:
# Inspect dataset structure

for folder in DATA.iterdir():
    if folder.is_dir():
        print(f"\n{folder.name}/")

        for class_folder in folder.iterdir():
            if class_folder.is_dir():
                count = len(list(class_folder.glob("*.jpg")))
                print(f"  └── {class_folder.name}/ : {count} images")


validation/
  └── 2_Pneumonia/ : 943 images
  └── 0_Normal/ : 654 images
  └── 1_Covid19/ : 256 images

train/
  └── 1_Covid19/ : 1025 images
  └── 2_Pneumonia/ : 3742 images
  └── 0_Normal/ : 2616 images


In [6]:
# Verify total and class counts

classes = ["0_Normal", "1_Covid19", "2_Pneumonia"]

total = 0
classes_count = {}
for c in classes:
    count = len(list(DATA.rglob(f"{c}/*.jpg")))
    classes_count.update({c: count})
    total += count
    print(f"{c}: {count}")

print("\nTotal images:", total)

0_Normal: 3270
1_Covid19: 1281
2_Pneumonia: 4685

Total images: 9236


In [7]:
# Verify train/validation counts

for split in ["train", "validation"]:
    print(f"\n{split.upper()}")

    total = 0

    for c in classes:
        count = len(list((DATA / split / c).glob("*.jpg")))
        total += count
        print(f"{c}: {count}")

    print("Total:", total)


TRAIN
0_Normal: 2616
1_Covid19: 1025
2_Pneumonia: 3742
Total: 7383

VALIDATION
0_Normal: 654
1_Covid19: 256
2_Pneumonia: 943
Total: 1853


In [8]:
from PIL import Image

files = list(DATA.rglob("*.jpg"))

chunk_size = 500

for start in range(0, len(files), chunk_size):

    chunk = files[start:start + chunk_size]

    dimensions = set()
    formats = set()
    modes = set()

    for f in chunk:
        with Image.open(f) as img:
            dimensions.add(img.size)
            formats.add(img.format)
            modes.add(img.mode)

    print(f"Chunk {start//chunk_size + 1}")
    print("Dimensions:", dimensions)
    print("Formats:", formats)
    print("Color modes:", modes)

Chunk 1
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 2
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 3
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 4
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 5
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 6
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 7
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 8
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 9
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 10
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 11
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 12
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 13
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 14
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color mo

In [9]:
# Inspect filename structure

for split in ["train", "validation"]:
    print(f"\n{split.upper()}")

    for c in classes:
        files = list((DATA / split / c).glob("*.jpg"))

        print(f"\n{c}:")
        for f in files[:5]:
            print(" ", f.name)


TRAIN

0_Normal:
  Normal (246).jpg
  Normal (2426).jpg
  Normal (2465).jpg
  Normal (2400).jpg
  Normal (2419).jpg

1_Covid19:
  COVID-19 (146).jpg
  COVID-19 (114).jpg
  COVID-19 (173).jpg
  COVID-19 (134).jpg
  COVID-19 (174).jpg

2_Pneumonia:
  Pneumonia-Bacterial (750).jpg
  Pneumonia-Bacterial (744).jpg
  Pneumonia-Bacterial (730).jpg
  Pneumonia-Bacterial (772).jpg
  Pneumonia-Bacterial (751).jpg

VALIDATION

0_Normal:
  Normal (2623).jpg
  Normal (2629).jpg
  Normal (2637).jpg
  Normal (2643).jpg
  Normal (2649).jpg

1_Covid19:
  COVID-19 (1068).jpg
  COVID-19 (1074).jpg
  COVID-19 (1051).jpg
  COVID-19 (1085).jpg
  COVID-19 (1046).jpg

2_Pneumonia:
  Pneumonia-Viral (1019).jpg
  Pneumonia-Viral (1016).jpg
  Pneumonia-Viral (1009).jpg
  Pneumonia-Viral (1007).jpg
  Pneumonia-Viral (1006).jpg


In [10]:
# Check class imbalance

for name, count in classes_count.items():
    percentage = count / total * 100
    print(f"{name}: {count} ({percentage:.2f}%)")

0_Normal: 3270 (176.47%)
1_Covid19: 1281 (69.13%)
2_Pneumonia: 4685 (252.83%)


In [11]:
# Inspect Pneumonia filename subcategories
from collections import Counter
pneumonia_files = list(DATA.rglob("2_Pneumonia/*.jpg"))

subcategories = Counter()

for f in pneumonia_files:
    name = f.name.lower()

    if "bacterial" in name:
        subcategories["Bacterial"] += 1
    elif "viral" in name:
        subcategories["Viral"] += 1
    else:
        subcategories["Unknown"] += 1

print(subcategories)

Counter({'Bacterial': 3001, 'Viral': 1684})


In [42]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# Dataset root
DATA_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Curated COVID-19 Chest X-Ray Dataset"
)

# Output location
OUTPUT = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "covid_chest_x-ray_curated_dataset/data_split.csv"
)

# Classes
classes = {
    "0_Normal": 0,
    "1_Covid19": 1,
    "2_Pneumonia": 2
}

# --------------------------------------------------
# 1. Collect images using RELATIVE paths
# --------------------------------------------------

rows = []

for class_name, label in classes.items():
    for image_path in DATA_ROOT.rglob(f"{class_name}/*.jpg"):

        relative_path = image_path.relative_to(DATA_ROOT)

        rows.append({
            "path": str(relative_path),
            "label": label,
            "class": class_name
        })

df = pd.DataFrame(rows)

print("Total images:", len(df))

Total images: 9236


In [40]:
# 2. Create stratified 70/15/15 split
# --------------------------------------------------

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

# Add split column
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

# Combine
split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

In [44]:
print("=== SPLIT SIZES ===")
print(split_df["split"].value_counts())

print("\n=== CLASS DISTRIBUTION ===")
print(
    pd.crosstab(
        split_df["split"],
        split_df["class"]
    )
)

print("\n=== CLASS PERCENTAGES ===")
print(
    (
        pd.crosstab(
            split_df["split"],
            split_df["class"],
            normalize="index"
        ) * 100
    ).round(2)
)

print("\n=== DUPLICATE CHECK ===")
print("Duplicate paths:", split_df["path"].duplicated().sum())

print("\n=== TOTAL CHECK ===")
print("Total:", len(split_df))
print("Expected:", len(df))
print("Correct:", len(split_df) == len(df))

print("\n=== SPLIT CHECK ===")
print("Splits:", sorted(split_df["split"].unique()))

print("\n=== CLASS CHECK ===")
print("Classes:", sorted(split_df["class"].unique()))

=== SPLIT SIZES ===
split
train         6465
test          1386
validation    1385
Name: count, dtype: int64

=== CLASS DISTRIBUTION ===
class       0_Normal  1_Covid19  2_Pneumonia
split                                       
test             491        192          703
train           2289        897         3279
validation       490        192          703

=== CLASS PERCENTAGES ===
class       0_Normal  1_Covid19  2_Pneumonia
split                                       
test           35.43      13.85        50.72
train          35.41      13.87        50.72
validation     35.38      13.86        50.76

=== DUPLICATE CHECK ===
Duplicate paths: 0

=== TOTAL CHECK ===
Total: 9236
Expected: 9236
Correct: True

=== SPLIT CHECK ===
Splits: ['test', 'train', 'validation']

=== CLASS CHECK ===
Classes: ['0_Normal', '1_Covid19', '2_Pneumonia']


In [45]:
missing = [
    path for path in split_df["path"]
    if not (DATA_ROOT / path).exists()
]

print("\n=== FILE CHECK ===")
print("Missing files:", len(missing))

if missing:
    print("\nFirst missing files:")
    for path in missing[:10]:
        print(path)


=== FILE CHECK ===
Missing files: 0


In [46]:
# --------------------------------------------------
# 3. Save FINAL portable CSV
# --------------------------------------------------

split_df.to_csv(OUTPUT, index=False)

print("\nSaved:", OUTPUT)
print("\nSplit distribution:")
print(split_df["split"].value_counts())

print("\nClass distribution:")
print(
    split_df.groupby(["split", "class"])
    .size()
)

print("\nFirst 5 rows:")
display(split_df.head())


Saved: /content/drive/MyDrive/Colab Notebooks/covid_chest_x-ray_curated_dataset/data_split.csv

Split distribution:
split
train         6465
test          1386
validation    1385
Name: count, dtype: int64

Class distribution:
split       class      
test        0_Normal        491
            1_Covid19       192
            2_Pneumonia     703
train       0_Normal       2289
            1_Covid19       897
            2_Pneumonia    3279
validation  0_Normal        490
            1_Covid19       192
            2_Pneumonia     703
dtype: int64

First 5 rows:


,path,label,class,split
0,train/0_Normal/Normal (1118).jpg,0,0_Normal,train
1,train/2_Pneumonia/Pneumonia-Viral (326).jpg,2,2_Pneumonia,train
2,train/2_Pneumonia/Pneumonia-Bacterial (1589).jpg,2,2_Pneumonia,train
3,train/1_Covid19/COVID-19 (782).jpg,1,1_Covid19,train
4,validation/0_Normal/Normal (3078).jpg,0,0_Normal,train
